In [1]:
import os
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import cv2
from tqdm import tqdm
import shutil

def is_image_file(filename):
    return any(filename.lower().endswith(ext) for ext in ['.png', '.jpg', '.jpeg'])

class BreakHisDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        subclasses = ['adenosis', 'fibroadenoma', 'tubular_adenoma', 'phyllodes_tumor',
                      'ductal_carcinoma', 'lobular_carcinoma', 'mucinous_carcinoma', 'papillary_carcinoma']
        
        for subclass in subclasses:
            subclass_path = os.path.join(root_dir, 'benign', 'SOB', subclass)
            if not os.path.exists(subclass_path):
                subclass_path = os.path.join(root_dir, 'malignant', 'SOB', subclass)
            
            if os.path.exists(subclass_path):
                for patient_folder in os.listdir(subclass_path):
                    patient_path = os.path.join(subclass_path, patient_folder)
                    if os.path.isdir(patient_path):
                        for mag_folder in os.listdir(patient_path):
                            mag_path = os.path.join(patient_path, mag_folder)
                            if os.path.isdir(mag_path):
                                for file in os.listdir(mag_path):
                                    file_path = os.path.join(mag_path, file)
                                    if is_image_file(file_path):
                                        self.image_paths.append(file_path)
                                        self.labels.append(subclasses.index(subclass))
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = BreakHisDataset(root_dir="/kaggle/input/breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast", transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True,pin_memory=True)

/usr/local/lib/python3.10/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.5 (you have 1.4.20). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
print(len(dataset))

7909


In [3]:

batch = next(iter(dataloader))
images, labels = batch

print(f"Batch size: {images.shape}")  
print(f"Labels: {labels[:10]}") 


Batch size: torch.Size([32, 3, 256, 256])
Labels: tensor([4, 2, 7, 2, 1, 0, 7, 4, 3, 7])


In [4]:
elastic_transform = A.ElasticTransform(
    alpha=2,   
    sigma=25, 
    alpha_affine=10,  
    p=1.0 


<ipython-input-4-b441c8d8dc5c>:1: UserWarning: Argument 'alpha_affine' is not valid and will be ignored.
  elastic_transform = A.ElasticTransform(


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [6]:
def apply_elastic_transform(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        image_np = np.array(image)

        if image_np is None or image_np.size == 0:
            print(f"Skipping {image_path} (Invalid image)")
            return None

        augmented = elastic_transform(image=image_np)

        deformed_image = augmented['image'] 

        return Image.fromarray(deformed_image)  

    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None


In [7]:
input_root = "/kaggle/input/breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast"
output_root = "/kaggle/working/elastic_breakhis"  

os.makedirs(output_root, exist_ok=True) 


In [8]:
def process_and_save_images(input_root, output_root):
    subclasses = ['adenosis', 'fibroadenoma', 'tubular_adenoma', 'phyllodes_tumor',
                  'ductal_carcinoma', 'lobular_carcinoma', 'mucinous_carcinoma', 'papillary_carcinoma']

    for subclass in subclasses:
        input_subclass_path = os.path.join(input_root, 'benign', 'SOB', subclass)
        if not os.path.exists(input_subclass_path):  
            input_subclass_path = os.path.join(input_root, 'malignant', 'SOB', subclass)
        
        if os.path.exists(input_subclass_path):
            output_subclass_path = os.path.join(output_root, subclass)
            os.makedirs(output_subclass_path, exist_ok=True)

            for patient_folder in tqdm(os.listdir(input_subclass_path), desc=f"Processing {subclass}"):
                patient_path = os.path.join(input_subclass_path, patient_folder)
                if os.path.isdir(patient_path):
                    for mag_folder in os.listdir(patient_path):
                        mag_path = os.path.join(patient_path, mag_folder)
                        if os.path.isdir(mag_path):
                            for file in os.listdir(mag_path):
                                file_path = os.path.join(mag_path, file)
                                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                                    deformed_image = apply_elastic_transform(file_path)
                                    
                                    output_file_path = os.path.join(output_subclass_path, f"deformed_{file}")
                                    deformed_image.save(output_file_path)


In [9]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

   


In [10]:
process_and_save_images(input_root, output_root)


Processing papillary_carcinoma: 100%|██████████| 6/6 [01:37<00:00, 16.23s/it]


In [11]:
image_path="/kaggle/working/elastic_breakhis/lobular_carcinoma"
def apply_elastic_transform(image_path):
    image = Image.open(image_path).convert("RGB")  
    image_np = np.array(image)  # Convert to NumPy

   
    augmented = elastic_transform(image=image_np)
    deformed_image = augmented['image']

   
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(image_np)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(deformed_image)
    axes[1].set_title("Elastic Transformed")
    axes[1].axis("off")

    plt.show()

    return Image.fromarray(deformed_image)  # Convert back to PIL


In [12]:
shutil.make_archive("/kaggle/working/deformed_breakhis", 'zip', output_root)
print("Zipped file saved as deformed_breakhis.zip")


Zipped file saved as deformed_breakhis.zip
